# Análisis estadístico de rendimiento — cache_caliente vs cache_frio (D.1)

Recrea exactamente el análisis del Bloque C.1/D4: Wilcoxon de rangos con
signo (pareado) + Cliff's delta entre los escenarios `cache_caliente` y
`cache_frio`, sobre el p95 de cada corrida individual de
`docs/mediciones/perf/k6-run*.json` (5 corridas = mínimo exigido por la guía).

Este notebook **no duplica la lógica estadística**: invoca
[`scripts/perf-analysis.py`](../../scripts/perf-analysis.py), la misma fuente
de verdad que `make docs` y que `docs/mediciones/perf/REPORT.md`. Al abrirlo
se ve la salida del **último run real** que produjo los números archivados.


## Por qué subprocess y no import

La interfaz canónica de `perf-analysis.py` es su CLI (paths de las corridas
en `argv`, cálculo agregado + gráfico SVG en `main()`). Importar sus
funciones desde el notebook acoplaría esta evidencia a nombres internos del
script y arriesgaría *drift* (el notebook seguiría llamando una firma que ya
cambió). Por eso se ejecuta como subproceso con el intérprete actual
(`sys.executable`): el notebook corre **exactamente el mismo comando** que
`make docs` y que el documentado en `REPORT.md` — una sola fuente de verdad.

**Reproducibilidad (D.2):** la única aleatoriedad del pipeline es el IC 95%
del p95 por *bootstrap* (remuestreo con reemplazo), con **semilla fija 42**
(`BOOTSTRAP_SEED = 42` en `perf-analysis.py`, mismo valor que el PRNG
`mulberry32` de `k6/libros-listado-test.js`). Sin semilla explícita esto sería
un default no determinista del lenguaje; con ella, el gráfico SVG es
reproducible byte a byte en datos (matplotlib solo varía ids/timestamp de
metadata por corrida).


In [1]:
import os
import subprocess
import sys
from pathlib import Path

def raiz_repo():
    d = Path(os.path.abspath("")).resolve()
    while not (d / "Makefile").is_file() and d != d.parent:
        d = d.parent
    return d

REPO = raiz_repo()
runs = sorted((REPO / "docs/mediciones/perf").glob("k6-run*.json"))
print("Corridas a analizar (%d):" % len(runs))
for r in runs:
    print("  -", r.name)

# encoding explicito: la salida trae UTF-8 (tildes) que cp1252 de
# Windows no decodifica; sin esto el readerthread muere (ver Fase EV-2).
resultado = subprocess.run(
    [sys.executable, "scripts/perf-analysis.py", *[str(r) for r in runs]],
    cwd=str(REPO),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print(resultado.stdout)
if resultado.returncode != 0:
    print("stderr:", resultado.stderr)
    raise SystemExit(f"perf-analysis.py fallo con exit code {resultado.returncode}")


Corridas a analizar (5):
  - k6-run1.json
  - k6-run2.json
  - k6-run3.json
  - k6-run4.json
  - k6-run5.json


[
  {
    "escenario": "cache_caliente",
    "n_peticiones": 9821,
    "media_ms": 29.841755324203238,
    "mediana_ms": 22.63276,
    "desviacion_tipica_ms": 29.863240353464796,
    "ic95_media_ms": [
      29.25112580955393,
      30.432384838852546
    ],
    "p50_ms": 22.63276,
    "p90_ms": 46.993288,
    "p95_ms": 65.602856,
    "p99_ms": 129.35735059999988,
    "tasa_error_5xx": 0.0,
    "throughput_req_s": 196.42
  },
  {
    "escenario": "cache_frio",
    "n_peticiones": 10006,
    "media_ms": 11.012139606636019,
    "mediana_ms": 10.3034895,
    "desviacion_tipica_ms": 3.407205773252325,
    "ic95_media_ms": [
      10.945378398839258,
      11.07890081443278
    ],
    "p50_ms": 10.3034895,
    "p90_ms": 15.0599245,
    "p95_ms": 17.126417,
    "p99_ms": 23.236596600000045,
    "tasa_error_5xx": 0.0,
    "throughput_req_s": 200.12
  }
]

--- Comparación pareada cache_caliente vs cache_frio (p95 por corrida) ---
p95 cache_caliente por corrida: [129.64, 41.57, 31.16, 32.53, 32

In [2]:
# Resumen legible de la comparación pareada (último bloque de la salida).
texto = resultado.stdout or ""
marca = "--- Comparación pareada"
if marca in texto:
    print(texto[texto.index(marca):].strip())
else:
    print("(el script no pudo emitir la comparación pareada -- ver stdout completo arriba)")


--- Comparación pareada cache_caliente vs cache_frio (p95 por corrida) ---
p95 cache_caliente por corrida: [129.64, 41.57, 31.16, 32.53, 32.5]
p95 cache_frio por corrida:     [18.39, 18.71, 15.69, 15.75, 16.61]
Wilcoxon (rangos con signo, pareado, método: scipy.stats.wilcoxon):
  estadístico = 0.0000
  p-valor     = 0.062500
  Sin diferencia estadísticamente significativa (p>=0.05)
Cliff's delta = -1.0000 (efecto grande; cache_caliente tiende a ser MÁS lento)

Gráfico guardado en docs/mediciones/perf/p95-comparacion-escenarios.svg
